#### # 1. Define Gold Customer Dimension Logical Query

In [0]:
# Pure relational SQL utilizing your clean, verified Silver schema headers
gold_query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.customer_number,
    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE
        WHEN LOWER(ci.gender) <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'N/A')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.created_date AS create_date
FROM workspace.silver.crm_customers ci
LEFT JOIN workspace.silver.erp_customer ca
    ON ci.customer_number = ca.customer_id
LEFT JOIN workspace.silver.erp_location la
    ON ci.customer_number = la.customer_id
"""
df = spark.sql(gold_query)

#### # 3. Commit Schema to Target Storage & Verify Results

In [0]:
# Save directly as a high-performance Gold Delta Table with atomic overwrite
df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_customers")

# Display a clean data sample preview to verify integrity
df.limit(10).display()